# MLP Implementation

In [ ]:
import torch
# 输入(batch_size, input_features)
x = torch.rand(3, 4)
# 参数
W = torch.rand(4, 5)
b = torch.rand(5)
# 输出
y = x @ W + b
print(y.shape)

In [ ]:
import torch.nn as nn
# 与torch的线性层比较
linear = nn.Linear(4, 5)
y1 = linear(x)
print(y1.shape)

In [ ]:
# linear & mlp from scratch
class Linear(nn.Module):
    def __init__(self, input_features, output_features):
        super().__init__()
        self.W = nn.Parameter(torch.rand(input_features, output_features))
        self.b = nn.Parameter(torch.rand(output_features))
    def forward(self, x):
        return x @ self.W + self.b

class Mlp(nn.Module):
    def __init__(self, input_features, hidden_features,dropout):
        super().__init__()
        self.c_fc = Linear(input_features, hidden_features)
        self.gelu = nn.GELU()
        self.c_proj = Linear(hidden_features, input_features)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x

In [ ]:
# test
mlp = Mlp(x.size(1), 8, 0)
total_params = sum(p.numel() for p in mlp.parameters())
print(f"Total parameters in MLP: {total_params}") # 4 * 8 + 8 + 8 * 4 + 4 = 76
y2 = mlp(x)
print(y2.size())

In [ ]:
# softmax & crossentropy from scratch
def softmax(x):
    exp_x = torch.exp(x)
    return exp_x / exp_x.sum(dim=1, keepdim=True)

def cross_entropy(logits, target):
    epsilon = 1e-9 
    prob = softmax(logits)
    prob = torch.clamp(prob, epsilon, 1.0-epsilon) # 避免取0或1
    return -torch.log(prob[range(len(target)), target]).mean()

# test, assuming logits = y2
import torch.nn.functional as F
logits = y2
prob = softmax(logits)
print(f'prob:{prob}')
target = torch.tensor([0, 1, 3]) # 指定标签
loss1 = cross_entropy(logits, target)
loss2 = F.cross_entropy(logits, target)
print(f'loss1={loss1}, loss2={loss2}')

In [ ]:
# loop
optimizer = torch.optim.SGD(mlp.parameters(), lr=0.01)
for epoch in range(1000):
    optimizer.zero_grad()
    logits = mlp(x)
    loss = cross_entropy(logits, target)
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        print(f'Epoch {epoch}, Loss: {loss.item():.4f}')

In [ ]:
# inference
with torch.no_grad():
    logits = mlp(x)
    pred = torch.argmax(logits, dim=1)
    print(f'Predicted classes: {pred}')